In [1]:
import pandas as pd

In [2]:
# 1. Importation de la table des contrats
df_contrats = pd.read_csv(
    "data/Contrat_clean.csv", # Remplace par le bon chemin/nom
    sep=",",                       # Parfois ";" si le fichier vient d'Excel
    encoding="utf-8",              # Mets "latin-1" si tu as des problèmes avec les accents
    parse_dates=["date_debut_situation", "date_fin_situation"] # Magique : reconvertit direct en format Date !
)

# 2. Importation de la table des sinistres
df_sinistres = pd.read_csv(
    "data/Sinistre_clean.csv",
    sep=",",
    encoding="utf-8",
    parse_dates=["date_survenance", "date_declaration", "date_cloture", "date_gestion"]
)

# 3. Petit contrôle rapide
print(f"Contrats chargés : {len(df_contrats)} lignes")
print(f"Sinistres chargés : {len(df_sinistres)} lignes")

C:\Users\taoro\AppData\Local\Temp\ipykernel_11040\2903118456.py:2: DtypeWarning: Columns (0: id2_sinistre_vhr, 1: id3_sinistre_base, 2: id3_sinistre_0km) have mixed types. Specify dtype option on import or set low_memory=False.
  df_contrats = pd.read_csv(


Contrats chargés : 301437 lignes
Sinistres chargés : 67452 lignes


In [3]:
# 1. Liste des colonnes contenant les IDs de sinistres (noms réels de ton df)
cols_sinistres = [
    'id1_sinistre_base', 'id1_sinistre_0km', 'id1_sinistre_vhr', 
    'id2_sinistre_base', 'id2_sinistre_0km', 'id2_sinistre_vhr',
    'id3_sinistre_base', 'id3_sinistre_0km', 'id3_sinistre_vhr'
]

# 2. Colonnes que l'on veut conserver pour l'analyse (tes id_vars)
id_vars_etude = [
    'id_contrat', 'date_debut_situation', 'date_fin_situation', 
    'age_conducteur', 'vehicule_marque', 'code_insee'
]

# 3. Transformation de la table Contrat (Format Large -> Format Long)
df_contrats_long = df_contrats.melt(
    id_vars=id_vars_etude,
    value_vars=cols_sinistres,
    var_name='type_garantie_source', # Origine du sinistre (base, 0km, etc.)
    value_name='idx_sin'             # L'ID du sinistre qui va servir de clé
)

# 4. Nettoyage : On supprime les lignes où il n'y a pas d'ID de sinistre (NaN)
df_contrats_long = df_contrats_long.dropna(subset=['idx_sin'])

# 5. Jointure finale avec la table Sinistres
# Note : Vérifie si dans ta table sinistre la colonne s'appelle 'idx_sin' ou 'idx_sin_sin'
df_final = pd.merge(
    df_sinistres, 
    df_contrats_long, 
    left_on='idx_sin', # Nom dans la table Sinistre
    right_on='idx_sin', 
    how='inner'
)

print(f"Analyse prête ! {len(df_final)} sinistres ont été rattachés à leurs contrats.")

KeyError: 'idx_sin'